# Task 3 — Leaderboard model

**Name of Machine Learning Model: Gradient-Boosted Decision Trees (LightGBM),
selected from a six-model comparison.**

Constraint: **no deep learning, no LLMs.** Every model here is a classical
classifier over the provided TF-IDF features.

The full pipeline lives in `best_model.py` (it takes ~3.5 h for all six models).
This notebook explains what it does, and reports what it found.

Run it with:

```bash
python3 best_model.py --fast          # smoke test
python3 best_model.py --trials 40     # full run
```


## 1. The problem, and what the data is telling us

The competition page states plainly:

> *"You are expected to achieve a high training F1 score, but low test F1
> score ... **DO NOT OVER-ENGINEER YOUR SOLUTION**."*

That is a **declared train/test distribution shift** produced by how the
GenAIDetect organisers sampled the data: the test set draws on generators and
domains the training set does not cover. Two consequences drove every design
decision below:

1. Heavy hyperparameter search on in-distribution CV will not transfer. Past a
   modest budget, extra tuning fits training-domain quirks.
2. The in-distribution winner is not necessarily the out-of-distribution
   winner. Where models are within noise of each other, the **simpler** one is
   the better bet.


## 2. How XGBoost / LightGBM work

Gradient boosting builds an additive ensemble of shallow regression trees. Each
new tree is fitted to the *gradient of the loss* with respect to the current
ensemble's predictions — so each tree corrects what the ensemble so far gets
wrong:

$$\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta \, f_t(x)$$

XGBoost's specific contributions are a **regularised objective** and a
**second-order (Newton) approximation** of the loss, using both gradient
$g_i$ and Hessian $h_i$:

$$\mathcal{L}^{(t)} \simeq \sum_i \left[ g_i f_t(x_i) + \tfrac{1}{2} h_i f_t(x_i)^2 \right] + \Omega(f_t),
\qquad \Omega(f) = \gamma T + \tfrac{1}{2}\lambda \lVert w \rVert^2$$

where $T$ is the leaf count. This gives a closed-form optimal leaf weight and a
split-gain criterion that already includes the regularisation penalty.

**LightGBM** differs mainly in growth strategy: it grows **leaf-wise** (always
splitting the leaf with the highest loss reduction) rather than level-wise. On
wide sparse data this reaches the same accuracy far faster — which turned out
to matter a great deal here.


## 3. Results — six models

Holdout Macro-F1 on a 15% slice **never seen** by hyperparameter search,
blending, or threshold calibration.

| Model | Holdout Macro-F1 | OOF | CV (selection-biased) | Fit time |
|---|---|---|---|---|
| xgboost  | **0.7529** | 0.7391 | 0.7123 ± 0.0103 | 5431 s |
| lightgbm | 0.7514 | 0.7421 | 0.7149 ± 0.0136 | **301 s** |
| logreg   | 0.7377 | 0.7322 | 0.7152 ± 0.0066 | 30 s |
| linsvc   | 0.7360 | 0.7316 | 0.7078 ± 0.0078 | 49 s |
| rforest  | 0.7078 | 0.6985 | 0.6931 ± 0.0111 | 4382 s |
| compnb   | 0.6773 | 0.6774 | 0.6761 ± 0.0039 | 22 s |
| **blend** | 0.7540 | 0.7490 | — | — |

`CV (selection-biased)` is Optuna's `study.best_value` — the **maximum** of many
noisy CV estimates, and therefore optimistic. It is shown for the tuning roadmap
only; the holdout column is the honest number.

Every `OOF − holdout` value is **negative**, i.e. no model is fitting fold
noise. That is the opposite of the classic overfitting signature.


## 4. Why the blend was rejected

The blend scored highest (0.7540). It was still discarded, on evidence:

- Blend holdout 95% CI: **[0.7382, 0.7697]**
- XGBoost holdout 95% CI: **[0.7365, 0.7688]**
- **McNemar's test:** 104 vs 113 discordant pairs, **p = 0.587**

The blend's 0.0011 lead is not distinguishable from noise, so the simpler single
model wins. (Method from Sujon et al. 2025, *Journal of Big Data* 12:268,
`doi:10.1186/s40537-025-01313-4`, which finds F1 the most stable metric under
class imbalance and recommends bootstrap CIs plus McNemar over bare point
estimates.)

**The blend's weights are more interesting than its score:**

| model | weight |
|---|---|
| lightgbm | 0.379 |
| rforest | 0.229 |
| linsvc | 0.217 |
| compnb | 0.080 |
| xgboost | 0.071 |
| logreg | 0.025 |

XGBoost gets only 0.071 *despite being the best single model*, because it is
nearly a duplicate of LightGBM. RandomForest and LinearSVC earn real weight
despite worse solo scores. **A blend rewards decorrelation, not individual
accuracy.**


## 5. Final choice: LightGBM

The automated one-standard-error rule selected XGBoost. **We override that.**

The rule found `within-1SE = [xgboost, lightgbm]` — 0.0015 apart against a
bootstrap SE of 0.0084, with near-identical confidence intervals — then broke
the tie on a complexity ordering that ranks xgboost below lightgbm. That picked
the model costing **18× the compute** for a difference well inside noise.

LightGBM is the better answer on every ground that matters: equal accuracy
within measurement error, an eighteenth of the training cost, and no reason
under distribution shift to believe XGBoost's extra 0.0015 survives contact with
the test set. The two submissions agree on 92.7% of rows.

**Submit `outputs/LGBM_leaderboard_predictions.csv`.**


In [ ]:
# Reproduce the recommended submission (~3 min).
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb

DATA = Path("..") / "data"; OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
SEED, THR = 50007, 0.540      # threshold calibrated on out-of-fold predictions

tr = pd.read_csv(DATA / "train_features.csv"); te = pd.read_csv(DATA / "test_features.csv")
y = tr["label"].to_numpy().astype(int)
feat = [c for c in tr.columns if c not in ("id", "label")]   # exclude the label!
X, Xt = tr[feat].to_numpy(np.float32), te[feat].to_numpy(np.float32)
spw = float((y == 0).sum() / (y == 1).sum())

PARAMS = dict(num_leaves=46, learning_rate=0.06304716097331702,
              min_child_samples=12, subsample=0.9969969063623275,
              colsample_bytree=0.7764196845461617,
              reg_lambda=23.38434482088143, reg_alpha=0.00012647265418366007)

# Seed averaging: pure variance reduction. It cannot overfit, because no
# decision is taken from validation data.
proba = np.zeros(len(Xt))
for s in range(3):
    m = lgb.LGBMClassifier(**PARAMS, objective="binary", n_estimators=400,
                           random_state=SEED + s, n_jobs=-1,
                           scale_pos_weight=spw, verbose=-1, subsample_freq=1)
    m.fit(X, y); proba += m.predict_proba(Xt)[:, 1] / 3

pred = (proba >= THR).astype(int)
pd.DataFrame({"id": te["id"], "label": pred}).to_csv(
    OUT / "LGBM_leaderboard_predictions.csv", index=False)
print("saved | predicted positive rate:", round(pred.mean(), 4),
      "| training positive rate:", round(y.mean(), 4))


## 6. Tuning roadmap

Search strategy: **Optuna TPE**, 10 trials per model, 4-fold stratified CV,
scored on Macro-F1 (the competition metric, not accuracy or logloss).

| Parameter | Search range | Why this range |
|---|---|---|
| `max_depth` | 3–8 | Text signal is largely additive; deeper memorises |
| `min_child_weight` | 1–40 (log) | Default 1 lets a leaf form on one rare token across 5000 sparse columns |
| `colsample_bytree` | 0.25–0.8 | Low floor decorrelates trees on wide TF-IDF |
| `reg_lambda` | 1–100 (log) | Safe to push hard; default 1 is too weak here |
| `subsample` | 0.6–1.0 | Cheap variance reduction |
| `learning_rate` | 0.02–0.25 (log) | Paired with early stopping |
| `n_estimators` | fixed 2000 + early stopping | **Learned, not searched** |

**Anti-overfitting controls**, all measured rather than assumed:

1. 15% holdout split off *before* any tuning; never seen by search, blending or
   thresholding.
2. `study.best_value` reported separately as selection-biased — it is the max of
   many noisy estimates.
3. Early stopping on boosters, so tree count is learned. (Measured
   `best_iteration ≈ 288`, so `n_estimators` 2000 vs 400 changes nothing but
   wall-clock.)
4. Blend weights and decision threshold fitted on **out-of-fold** predictions,
   then validated on the untouched holdout. The gap between the two is the
   overfitting readout.
5. Seed averaging on the final fit.

**Cost control.** Measured per-fit cost at 20k×5000: xgboost ~270 s, lightgbm
~98 s, rforest ~87 s, linsvc ~12 s, logreg ~8 s, compnb ~5 s. At the original
budget that was ~15 h for the zoo. Hyperparameter *rankings* are far more stable
across sample size than the scores are, so the search runs on a 7000-row
stratified subsample (`--tune-subsample`) while OOF, holdout and the final fit
all use full data — no reported number comes from the subsample.


## 7. What did not work, and why that matters

- **RandomForest: 4382 s to finish last (0.7078).** Bagging over 5000 sparse
  TF-IDF features loses to boosting and to plain linear models alike.
- **Threshold calibration bought nothing at the top of the table** — the OOF
  sweep returned exactly 0.500 for XGBoost. It did move for others (LightGBM
  0.540, LinearSVC 0.565), so the machinery works; this data just did not need
  it for the winner.
- **The blend, despite the best raw score** — rejected by McNemar (see §4).
- **Label leakage, caught during development.** `train_features.csv` is
  `[id, label, 0001..5000]`. An earlier loader dropped only the `id` column,
  leaving `label` in `X` as a feature. It surfaced only because PCA raised a
  5001-vs-5000 shape mismatch; had the shapes agreed it would have silently
  produced a fake-perfect score. Feature columns are now taken as the
  intersection with the test set, which cannot contain the label.

**Expect the leaderboard score to sit below the 0.75 holdout figure.** The
competition page says so explicitly. That gap is designed into the dataset and
is not something to tune away.
